In [1]:
import pandas as pd
import numpy as np
import pickle

from ads.dataset.factory import DatasetFactory
from ads.dataset.dataset_browser import DatasetBrowser
from time import process_time

import ads
import oci

In [2]:
# [Parameter:oci_obj] OCI Object Storage
par_oci_obj_bucket_name            = 'data-science-source'
# The profile parameter (ociProfileName) 'LOCAL' or 'DATAFLOW' in OCI
par_oci_obj_ociConfigFilePath      = '~/oci/config'
par_oci_obj_ociProfileName         = 'DEFAULT'
par_oci_obj_config                 = oci.config.from_file(par_oci_obj_ociConfigFilePath, par_oci_obj_ociProfileName)
par_oci_obj_object_storage_client  = oci.object_storage.ObjectStorageClient(par_oci_obj_config)
par_oci_obj_namespace_name         = par_oci_obj_object_storage_client.get_namespace().data

In [3]:
# fn_loaded_data_to_pandas
# Descripción : Cargar datos de OCI Object Storage y convertir de Dataset a Dataframe (Pandas)
# Referencia  : https://docs.oracle.com/en-us/iaas/tools/ads-sdk/latest/user_guide/loading_data/connect.html?highlight=cket_name%20namespace%20file_name
def fn_loaded_data_to_pandas(object_name):
    try:
        ts = process_time()
        ads.set_auth(auth='resource_principal')
        storage_options = {
           "config"  : par_oci_obj_ociConfigFilePath,
           "profile" : par_oci_obj_ociProfileName
        }
        ds = ads.dataset.factory.DatasetFactory.open(f"oci://{par_oci_obj_bucket_name}@{par_oci_obj_namespace_name}/{object_name}", storage_options=storage_options)
        df = ds.to_pandas_dataframe()

        te = process_time()
        print('# Loaded data to pandas (' + object_name + ')...[seg: '+ str(te-ts) +']')
        
        return df
    
    except Exception as e:
        print(e)   

In [4]:
# [Exanple] fn_loaded_data_to_pandas
df = fn_loaded_data_to_pandas('uso_historico_giro_o_trans_mismo_hb_6.csv')
df

loop1:   0%|          | 0/4 [00:00<?, ?it/s]

# Loaded data to pandas (uso_historico_giro_o_trans_mismo_hb_6.csv)...[seg: 1.7934989619999993]


,Unnamed:_0,CUENTA,IMPORTE,IMPORTE_MEDIANA,IMPORTE_PROMEDIO,IMPORTE_Q1,IMPORTE_Q3,OPE,OPE_MEDIANA,OPE_PROMEDIO,OPE_Q1,OPE_Q3,PERIODO
0,0,8.400431e+10,2353.38,456.245,588.345000,384.1750,660.4150,15,3.5,3.750000,2.50,4.75,202103
1,1,8.400443e+10,52.80,52.800,52.800000,52.8000,52.8000,1,1.0,1.000000,1.00,1.00,202103
2,2,8.400481e+10,560.00,120.000,140.000000,114.6250,145.3750,6,1.0,1.500000,1.00,1.50,202103
3,3,8.400512e+10,1000.00,1000.000,1000.000000,1000.0000,1000.0000,1,1.0,1.000000,1.00,1.00,202103
4,4,8.400630e+10,344.21,172.105,172.105000,111.0525,233.1575,3,1.5,1.500000,1.25,1.75,202103
...,...,...,...,...,...,...,...,...,...,...,...,...,...
345674,345674,1.011606e+11,150.00,150.000,150.000000,150.0000,150.0000,1,1.0,1.000000,1.00,1.00,202106
345675,345675,1.011611e+11,249.98,64.990,83.326667,42.4950,114.9900,7,2.0,2.333333,1.50,3.00,202106
345676,345676,1.011614e+11,300.00,150.000,150.000000,135.0000,165.0000,2,1.0,1.000000,1.00,1.00,202106
345677,345677,1.011795e+11,3534.00,1767.000,1767.000000,1150.5000,2383.5000,2,1.0,1.000000,1.00,1.00,202106


In [6]:
# fn_upload_object
# Descripción : Subir objecto a OCI Object Storage y convertir de Dataset a Bytes (CSV)
# Referencia  : https://github.com/jganggini/oci-data-flow/blob/main/upload-unstructured-data-to-autonomous-database/src/oci_object_storage.py
def fn_upload_object(bucket_name, object_name, obj_bytes):
    try:
        ts = process_time()
        par_oci_obj_object_storage_client.put_object(namespace_name=par_oci_obj_namespace_name,bucket_name=bucket_name, object_name=object_name, put_object_body=obj_bytes)
        te = process_time()
        
        print('# Upload object (' + object_name + ')...[seg: '+ str(te-ts) +']')

    except Exception as e:
        print(e)

In [7]:
# [Exanple] fn_upload_object
obj_bytes = bytes(df.to_csv(line_terminator='\r\n', index=False), encoding='utf-8')
fn_upload_object(par_oci_obj_bucket_name, 'target/project_name/file_name.csv', obj_bytes)

# Upload object (target/project_name/file_name.csv)...[seg: 0.05448534299999963]


In [8]:
# fn_read_object
# Descripción : Descarga objeto de OCI Object Storage (Bytes)
# Referencia  : https://github.com/jganggini/oci-data-flow/blob/main/upload-unstructured-data-to-autonomous-database/src/oci_object_storage.py
def fn_download_object(bucket_name, object_name):
    try:
        ts = process_time()
        get_object_response = par_oci_obj_object_storage_client.get_object(namespace_name=par_oci_obj_namespace_name,bucket_name=bucket_name, object_name=object_name)
        te = process_time()
        print('# Download object (' + object_name + ')...[seg: '+ str(te-ts) +']')
        
        return get_object_response

    except Exception as e:
        print('  Download object (' + object_name + ') not exists...[Warning]')

        return e

In [9]:
# [Exanple] fn_download_object
obj_response = fn_download_object(par_oci_obj_bucket_name, 'Prueba2/dataprueba.txt')

# Decode Object
ts = process_time()
obj_bytes = obj_response.data.content
obj_str = obj_bytes.decode('latin-1')
te = process_time()
print('# Decode object...[seg: '+ str(te-ts) +']')

# Filter Object
ts = process_time()
lis_old = obj_str.split("\r\n")
lis_new = list(filter(lambda item : len(item)>55 , lis_old))
te = process_time()
print('# Filter object...[seg: '+ str(te-ts) +']')

# Dataframe
df = pd.DataFrame(lis_new)
df

# Download object (Prueba2/dataprueba.txt)...[seg: 0.008487694999999462]
# Decode object...[seg: 30.050673869999997]
# Filter object...[seg: 26.541993870000006]


,0
0,10038518267|20210930| ||1|29409477|1| |002|00000|00000|00000|00000|10000||BAUTISTA||JUANA|
1,10029280002|20210930| ||1|06146421|1| |001|00000|00000|00000|00000|10000||BUSTAMANTE||JOSE|LUIS
2,10016584398|20210930| ||1|07699936|1| |001|00000|00000|00000|00000|10000||CASTILLO||ARMANDO|NOLBERTO
3,10050957942|20210930| ||1|00478547|1| |001|00000|00000|00000|00000|10000||CUADROS DE AYCA||MARIA|HILDA
4,10015096683|20210930| ||1|06210773|1| |001|00000|00000|00000|00000|10000||LEYVA||ANA|MARIA
...,...
11301995,10000497088|20210930|3|20101004610|||2| |001|00000|00000|00000|00000|10000|ÑUSTA TRAVEL SERVICE S A||||
11301996,10139552431|20210930|3|20539251971|||2| |001|10000|00000|00000|00000|00000|ÑUÑURCO TRAVELLERS SERVICIOS TURISTICOS E.I.R.L||||
11301997,10107420231|20210930| ||1|44238764|1| |001|10000|00000|00000|00000|00000|ÖZGÜRLER|CASTILLO||EMIR|MEHDI
11301998,10081728950|20210930| ||1|45419390|1| |002|10000|00000|00000|00000|00000|ÖZGÜRLER|CASTILLO||ERMIS|BILGE


In [10]:
# [Exanple] Dataframe Split column
df = df[0].str.split('|',expand=True)
te = process_time()
print('# Split column...[seg: '+ str(te-ts) +']')
df

# Split column...[seg: 137.289145706]


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,10038518267,20210930,,,1,29409477,1,,002,00000,00000,00000,00000,10000,,BAUTISTA,,JUANA,
1,10029280002,20210930,,,1,06146421,1,,001,00000,00000,00000,00000,10000,,BUSTAMANTE,,JOSE,LUIS
2,10016584398,20210930,,,1,07699936,1,,001,00000,00000,00000,00000,10000,,CASTILLO,,ARMANDO,NOLBERTO
3,10050957942,20210930,,,1,00478547,1,,001,00000,00000,00000,00000,10000,,CUADROS DE AYCA,,MARIA,HILDA
4,10015096683,20210930,,,1,06210773,1,,001,00000,00000,00000,00000,10000,,LEYVA,,ANA,MARIA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11301995,10000497088,20210930,3,20101004610,,,2,,001,00000,00000,00000,00000,10000,ÑUSTA TRAVEL SERVICE S A,,,,
11301996,10139552431,20210930,3,20539251971,,,2,,001,10000,00000,00000,00000,00000,ÑUÑURCO TRAVELLERS SERVICIOS TURISTICOS E.I.R.L,,,,
11301997,10107420231,20210930,,,1,44238764,1,,001,10000,00000,00000,00000,00000,ÖZGÜRLER,CASTILLO,,EMIR,MEHDI
11301998,10081728950,20210930,,,1,45419390,1,,002,10000,00000,00000,00000,00000,ÖZGÜRLER,CASTILLO,,ERMIS,BILGE


In [11]:
# [Exanple] Pickle
obj_response = fn_download_object(par_oci_obj_bucket_name, 'model_lgbm_cr_new_v6.sav')

# Decode Object
ts = process_time()
obj_bytes = obj_response.data.content

# Pickle
import pickle

mod_pickle = pickle.loads(obj_bytes)
mod_pickle
te = process_time()
print('# Read pickle object...[seg: '+ str(te-ts) +']')

# Download object (model_lgbm_cr_new_v6.sav)...[seg: 0.026180861000000277]
# Read pickle object...[seg: 0.7958363380000151]
